In [184]:
import math
import random

In [185]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(
            self.data + other.data,
            (self, other),
            '+'
        )

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward

        return out

    def __mul__(self, other):
        other =  other if isinstance(other, Value) else Value(other)
        out = Value(
        self.data * other.data,
        (self, other),
        '*'

    )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out
    def __rmul__(self, other):
        return self * other

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(
        self.data - other.data,
        (self, other),
        '-'
    )

        def _backward():
            self.grad += out.grad
            other.grad -= out.grad

        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other * self**-1
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad

        out._backward = _backward

        return out
    def loss_fn(prediction, target):
        return (prediction - target)**2
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


In [186]:
from email import errors


class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(
            self.data + other.data,
            (self, other),
            '+'
        )

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward

        return out

    def __mul__(self, other):
        other =  other if isinstance(other, Value) else Value(other)
        out = Value(
        self.data * other.data,
        (self, other),
        '*'

    )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out
    def __rmul__(self, other):
        return self * other

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(
        self.data - other.data,
        (self, other),
        '-'
    )

        def _backward():
            self.grad += out.grad
            other.grad -= out.grad

        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other * self**-1
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad

        out._backward = _backward

        return out
  
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


In [187]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)

        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

In [188]:
class Layer:
    def __init__(self, nin, nout, nonlin=True):
        self.neurons = [Neuron(nin, nonlin) for _ in range(nout)]

    def __call__(self, x):
        outs = [neuron(x) for neuron in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [189]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts

        self.layers = [
            Layer(
                sz[i],
                sz[i+1],
                nonlin=(i != len(nouts)-1)
            )
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [190]:
def mse_loss(predictions, targets):
        errors = [(pred - target)**2
        for pred, target in zip(predictions, targets)]

        return sum(errors , Value(0)) / len(errors)

In [191]:
c = Value(2.0) + Value(3.0)
c.data

5.0

In [192]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

result = a * b + c


result.backward()

print(result.data)
print(a.grad)
print(b.grad)
print(c.grad)

10.0
3.0
2.0
1.0


In [193]:
a = Value(5.0)
b = a * 3

print(b.data)  # 15.0

15.0


In [194]:
a = Value(2.0)
b = Value(3.0)

c = a * b + a * b + 5
c.backward()

print(c.data)
print(a.grad)
print(b.grad)

17.0
6.0
4.0


In [195]:
a = Value(3.0)
b = a**2

b.backward()
print(b.data)
print(a.grad)

9.0
6.0


In [196]:
a = Value(10.0)
b = Value(2.0)

c = a / b

c.backward()

print(c.data)
print(a.grad)
print(b.grad)

5.0
0.5
-2.5


In [197]:
a = Value(0.5)
b = a.tanh()

b.backward()

print(b.data)
print(a.grad)

0.46211715726000974
0.7864477329659274


In [198]:
a = Value(4.0)
b = a.relu()

b.backward()

print(b.data)
print(a.grad)

4.0
1.0


In [199]:
# 1. Create a neuron with 3 inputs
n = Neuron(3)

# 2. Define the input values
x = [Value(2.0), Value(3.0), Value(-1.0)]

# 3. Pass inputs through the neuron
output = n(x)

# 4. Print output
print("Output:", output.data)


# 5. Backpropagation
output.backward()

# 6. Inspect gradients
print("Output gradient:", output.grad)
print("Weight gradients:", [w.grad for w in n.w])
print("Bias gradient:", n.b.grad)

Output: -0.11940607456795385
Output gradient: 1.0
Weight gradients: [1.9714843787125444, 2.957226568068817, -0.9857421893562722]
Bias gradient: 0.9857421893562722


In [200]:
layer = Layer(3, 4)

x = [Value(2.0), Value(3.0), Value(-1.0)]

outputs = layer(x)

print("Number of neurons:", len(layer.neurons))
print("Number of outputs:", len(outputs))
print("Outputs:", [o.data for o in outputs])

Number of neurons: 4
Number of outputs: 4
Outputs: [-0.42660777635549774, -0.5074776253955996, 0.782709236174774, 0.32918121956676055]


In [201]:
model = MLP(3, [4, 4, 1])

x = [Value(2.0), Value(3.0), Value(-1.0)]

output = model(x)

print("Prediction:", output.data)

Prediction: -0.08262616019051505


In [202]:
output.backward()

for i, layer in enumerate(model.layers):

    print(f"\nLayer {i}")

    for j, neuron in enumerate(layer.neurons):

        print(
            f"Neuron {j} gradients:",
            [w.grad for w in neuron.w]
        )


Layer 0
Neuron 0 gradients: [-0.0376991872745795, -0.056548780911869254, 0.01884959363728975]
Neuron 1 gradients: [0.22693390680146083, 0.34040086020219124, -0.11346695340073042]
Neuron 2 gradients: [-0.07860164788758331, -0.11790247183137496, 0.039300823943791655]
Neuron 3 gradients: [-0.0008206180435474496, -0.0012309270653211744, 0.0004103090217737248]

Layer 1
Neuron 0 gradients: [-0.004718507115534952, -0.0030350648094123953, 0.00404788657918172, -0.004978131384377103]
Neuron 1 gradients: [0.18131109829836284, 0.11662394917017826, -0.15554215422125794, 0.19128729631525257]
Neuron 2 gradients: [0.0023912611586745587, 0.0015381205146248978, -0.002051401791818613, 0.0025228344327483717]
Neuron 3 gradients: [-0.14808024451867147, -0.09524901162660403, 0.12703425463863488, -0.1562279963969196]

Layer 2
Neuron 0 gradients: [-0.9360792678006453, -0.6300457073291562, 0.768190151480196, -0.4024169901581282]


In [203]:
prediction = Value(7.0)
target = Value(10.0)

loss = (prediction - target)**2

loss.backward()

print(loss.data)
print(prediction.grad)
print(target.grad)

9.0
-6.0
6.0


In [204]:
model = MLP(3, [4, 4, 1])

params = model.parameters()

print("Total parameters:", len(params))
print("Parameter values:", [p.data for p in params])

Total parameters: 41
Parameter values: [0.7559457525572892, -0.9294226017404932, -0.515059801726854, 0, 0.05069781417800523, 0.47308492890065734, 0.9618540988445754, 0, 0.5680644781096995, -0.24429199850955996, -0.3450049743928849, 0, -0.25605237705154793, -0.884766645017502, -0.6052948279537449, 0, -0.20761568646091977, -0.6081622506252164, -0.7650350345137193, -0.16998778611548704, 0, 0.886197408367714, 0.1408233502837415, -0.38762753988217025, 0.2530394721270135, 0, -0.182213397743898, -0.37867264516235544, 0.3310527741558309, -0.9546042077214683, 0, -0.1516341672577275, -0.8132010464954817, -0.02704358114963279, -0.8158477680223748, 0, 0.8610787886524311, 0.7298149798789297, 0.8179629130758894, 0.765642582764009, 0]


In [205]:
xs = [[Value(1.0)], [Value(2.0)], [Value(3.0) ], [Value(4.0)]]  
ys = [3.0 , 5.0, 7.0, 9.0]  # Example target values for a linear relationship

In [206]:
model = MLP(1, [4, 4, 1])  # MLP with 1 input and 1 output


In [210]:
for epoch in range(100):

    # 1. Forward propagation
    ypred = [model(x) for x in xs]

    # 2. Calculate loss
    loss = mse_loss(ypred, ys)

    # 3. Reset gradients
    for p in model.parameters():
        p.grad = 0.0

    # 4. Backpropagation
    loss.backward()

    # 5. Gradient descent
    for p in model.parameters():
        p.data -= 0.01 * p.grad

    # 6. Monitor training
    if epoch % 10 == 0:
        print(epoch, loss.data)



0 0.051962954905824664
10 0.05151855662144785
20 0.05108239861659475
30 0.05065397868781009
40 0.050232821174878976
50 0.04981847556449402
60 0.04941051518779162
70 0.04900853600505578
80 0.04861215547135135
90 0.048221011477257006


In [211]:
for x, y in zip(xs, ys):
    prediction = model(x)
    print(
        f"Input: {x[0].data}, "
        f"Target: {y}, "
        f"Prediction: {prediction.data:.4f}"
    )

Input: 1.0, Target: 3.0, Prediction: 2.9842
Input: 2.0, Target: 5.0, Prediction: 4.7013
Input: 3.0, Target: 7.0, Prediction: 6.9407
Input: 4.0, Target: 9.0, Prediction: 8.6864
